# Core Objective: 
1. Fetching the BrightData Profile 
    a. Validating the data. 
2. Fetching the prompt 
    a. Adding the inputs from the raw files. 
3. Generating the response. 
    a. Manually, checking the response generated by the LLM.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/gautamw_7/Desktop/Projects/job-search-assistant")

sys.path.insert(0, str(PROJECT_ROOT))

In [33]:
import time
import requests
import os

DATASET_ID = "gd_l1viktl72bvl7bjuj0"
API_KEY = os.getenv("BRIGHTDATA_API_KEY")

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

inputs = [
        {"url": "https://www.linkedin.com/in/ma-zeller/"},
        {"url": "https://www.linkedin.com/in/sasha-chavan-658068258/"},
        {"url": "https://www.linkedin.com/in/mohammed-taher-13934a51/"}
]

# Trigger the async collection
trigger_response = requests.post(
    "https://api.brightdata.com/datasets/v3/trigger",
    params={
        "dataset_id": DATASET_ID,
        "format": "json",
    },
    headers=headers,
    json=inputs,
)
trigger_response.raise_for_status()

snapshot_id = trigger_response.json()["snapshot_id"]
print("Snapshot ID:", snapshot_id)

# Step 2: Poll progress until ready
while True:
    progress_response = requests.get(
        f"https://api.brightdata.com/datasets/v3/progress/{snapshot_id}",
        headers={"Authorization": f"Bearer {API_KEY}"},
    )
    progress_response.raise_for_status()

    status = progress_response.json().get("status")
    print("Status:", status)

    if status == "ready":
        break
    if status == "failed":
        raise Exception(f"Collection failed: {progress_response.text}")

    time.sleep(10)

# Step 3: Download results
results_response = requests.get(
    f"https://api.brightdata.com/datasets/v3/snapshot/{snapshot_id}",
    params={"format": "json"},
    headers={"Authorization": f"Bearer {API_KEY}"},
)
results_response.raise_for_status()

Snapshot ID: sd_mtd308e6qg29w8an0
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: ready


In [34]:
def process_enrich_data(user_data, indent=0, enrich_string: str = "") -> str:
    """
    Recurisively prints a nested dictionary in a tree-like format.
    Return:
        enrich_str: The enriched string representation of the nested dictionary.
    """
    if not isinstance(user_data, dict):
        enrich_string += (" " * indent + str(user_data)) + "\n"
        return enrich_string

    for key, value in user_data.items():
        enrich_string += (" " * indent + str(key) + ":") + "\n"

        if isinstance(value, dict):
            process_enrich_data(value, indent + 4, enrich_string)
        elif isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    process_enrich_data(item, indent + 4, enrich_string)
                else:
                    enrich_string += (" " * (indent + 4) + str(item)) + "\n"
        else:
            enrich_string += (" " * (indent + 4) + str(value)) + "\n"

    return enrich_string


results = results_response.json()
print(f"Collected {len(results)} profiles")

Collected 3 profiles


In [55]:
import yaml

outreach_prompt_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/prompts/outreach_message.yaml"
with open(outreach_prompt_path, "r") as f:
    outreach_prompt = yaml.safe_load(f)

print(outreach_prompt)

system_prompt = outreach_prompt["system_prompt"]
user_prompt_template = outreach_prompt["user_prompt_template"]
print(user_prompt_template)

input_data = {}

{'name': 'outreach_generation', 'version': 2, 'purpose': 'Generate a personalized outreach message using the provided user,\nrecipient, context, and preferences.\n', 'system_prompt': "You are an assistant helping the user write a personalized outreach message.\n\nUse ONLY the information provided in the input.s\nNever invent facts, experiences, relationships, achievements, or context.\n\nFirst understand:\n1. Who the recipient is.\n2. Why the user wants to contact them.\n3. What information is genuinely relevant.\n4. What the user wants from the recipient.\n5. What tone and style are appropriate for the situation.\n\nThen write ONE natural, personalized message.\n\nThe message should:\n- Be clear and concise.\n- Sound natural rather than artificially polished.\n- Match the context and relationship between the user and recipient.\n- Use specific information when it is genuinely relevant.\n- Avoid unnecessary compliments and generic statements.\n- Avoid corporate buzzwords unless appropr

In [56]:
import pymupdf4llm

resume_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/data/Gautam_Warvatkar_Resume.pdf"
markdown_text = pymupdf4llm.to_markdown(resume_path)

In [57]:
import re
# 1. Check for count of text
if len(markdown_text) < 100:
    raise ValueError("Resume text is too short. Please check the resume file.")

# 2. Check for email
email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
if not re.search(email_pattern, markdown_text):
    raise ValueError("No email found in the resume. Please check the resume file.")

# 3. Check for phone number
phone_pattern = r"\+?\d[\d\s-]{7,}\d"
if not re.search(phone_pattern, markdown_text):
    raise ValueError("No phone number found in the resume. Please check the resume file.")

print(markdown_text)
input_data['user_resume'] = markdown_text

� Gautam Warvatkar 

� gautam.warvatkar@gmail.com 

# **Gautam Warvatkar** 

� +91 9987171860 

� Gautamw7 

## **Technical Skills** 

## **Languages:** Python, SQL 

**Machine Learning:** XGBoost, Scikit-learn, TensorFlow/Keras, CNNs, feature engineering, predictive modeling **AI / NLP:** RAG, Sentence Transformers, Gemini, embeddings, LLM re-ranking **Data Engineering:** Pandas, NumPy, ETL pipelines, FAISS, Qdrant, MongoDB **AWS:** S3, SageMaker Pipelines, Redshift, AWS Glue, boto3, Lambda, KMS **Tools:** Git, Jira, Confluence, Power Automate, Smartsheet 

## **Experience** 

**Data Scientist / Engineer Intern** Feb 2026 – Aug 2026 _Digital Productivity & Innovation, Syngenta Pune, India_ 

- **Supported transfer decisions across 11 distribution centers and runs covering up to $75M/day in product value** by engineering an explainable multi-gate Go/Caution/Stop decision framework using sales orders, inventory, historical consumption, and demand signals, targeting an official $6M/year 

In [58]:
user_communication_style_yaml_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/prompts/user_communication.yaml"

with open(user_communication_style_yaml_path, "r") as f:
    user_communication_style = yaml.safe_load(f)
input_data["user_communication_style"] = user_communication_style["default"]

In [59]:
user_preferences_yaml_path = "/home/gautamw_7/Desktop/Projects/job-search-assistant/prompts/user_preferences.yaml"
with open(user_preferences_yaml_path, "r") as f:
    user_preferences = yaml.safe_load(f)

input_data["user_preferences"] = user_preferences["default"]

In [60]:
input_data.keys()

dict_keys(['user_resume', 'user_communication_style', 'user_preferences'])

In [ ]:
prompts = []
for i, result in enumerate(results):
    input_data['recipient_enrich_data'] = process_enrich_data(result)
    user_note = None
    if i == 0:
        user_note = "So ask him, I very very much interseted in working with you, I'm currently learning German at B1 level, because of the carrer and the hope to shifting to Deutschland in future, so please consider me for any upcoming junior role. Vielen Dank"
    input_data['user_notes'] = user_note if user_note else ""
    formatted_prompt_template = user_prompt_template.format(**input_data)
    prompt = system_prompt + "\n" + formatted_prompt_template
    prompts.append(prompt)


['You are an assistant helping the user write a personalized outreach message.\n\nUse ONLY the information provided in the input.s\nNever invent facts, experiences, relationships, achievements, or context.\n\nFirst understand:\n1. Who the recipient is.\n2. Why the user wants to contact them.\n3. What information is genuinely relevant.\n4. What the user wants from the recipient.\n5. What tone and style are appropriate for the situation.\n\nThen write ONE natural, personalized message.\n\nThe message should:\n- Be clear and concise.\n- Sound natural rather than artificially polished.\n- Match the context and relationship between the user and recipient.\n- Use specific information when it is genuinely relevant.\n- Avoid unnecessary compliments and generic statements.\n- Avoid corporate buzzwords unless appropriate.\n- Avoid exaggerated claims.\n- Never invent personalization.\n- Make the requested action clear and easy to respond to.\n\nUse the user\'s communication style as a subtle infl

In [70]:
for i, prompt in enumerate(prompts):
    print(f"Prompt for profile {i}:")
    print(prompt)

Prompt for profile 0:
You are an assistant helping the user write a personalized outreach message.

Use ONLY the information provided in the input.s
Never invent facts, experiences, relationships, achievements, or context.

First understand:
1. Who the recipient is.
2. Why the user wants to contact them.
3. What information is genuinely relevant.
4. What the user wants from the recipient.
5. What tone and style are appropriate for the situation.

Then write ONE natural, personalized message.

The message should:
- Be clear and concise.
- Sound natural rather than artificially polished.
- Match the context and relationship between the user and recipient.
- Use specific information when it is genuinely relevant.
- Avoid unnecessary compliments and generic statements.
- Avoid corporate buzzwords unless appropriate.
- Avoid exaggerated claims.
- Never invent personalization.
- Make the requested action clear and easy to respond to.

Use the user's communication style as a subtle influence.

In [71]:
from google import genai

client = genai.Client()

responses = []

for i, prompt in enumerate(prompts):
    print(f"\n{'=' * 80}")
    print(f"GENERATING PROFILE {i}")
    print(f"{'=' * 80}")

    try:
        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt,
        )

        responses.append(response)

        print("RESPONSE:")
        print(response.text)

    except Exception as e:
        print(f"Error generating profile {i}: {e}")
        responses.append(None)


GENERATING PROFILE 0
RESPONSE:
Hey Marc,

I’m finishing my B.Tech in AI & Data Science and working as a Data Scientist/Engineer Intern, focusing on practical ML workflows, AWS ETL pipelines, and RAG systems.

I’m very interested in your applied ML work at AI.Associates. I’m currently learning German (B1 level) with plans to relocate to Germany, and I would love the opportunity to work with you.

If you have any upcoming junior ML or data engineering roles, please keep me in mind. I'd welcome the chance to connect.

Vielen Dank,
Gautam

GENERATING PROFILE 1
RESPONSE:
Hey Sasha,

I’m Gautam, a Data Science and AI engineering student in Pune with experience building data pipelines and ML systems at Syngenta. 

I noticed you're at Persistent Systems and wanted to reach out to connect with fellow tech folks in the city. 

Would love to connect here if you're open to it!

Gautam

GENERATING PROFILE 2
RESPONSE:
Hey Mohammed,

I saw your work as a Senior Data Scientist at Roche. I am finishin